# Heatmap

## Import

In [ ]:
# nur nutzen, wenn man Änderungen in Visualisierungen gemacht hat, aber Kernel nicht neu starten will.
import importlib
import visualisierungen

importlib.reload(visualisierungen)

In [ ]:
from visualisierungen import heatmap
import pandas as pd

## Heatmap all

In [ ]:
df = pd.read_csv("../data/processed/df_heatmap_with_positions.csv")
df.head()

In [ ]:
df = df.set_index("partei")

M = df.astype(float)
df_norm = (M-M.min().min())/(M.max().max()-M.min().min())
df_norm

In [ ]:
cantons = [str(c)[:2].upper() for c in df_norm.columns]

def party_short(name):
    s = str(name)
    if "-" not in s or "_" not in s:
        return s
    pre, rest = s.split("-", 1)
    mid = rest.split("_", 1)[0]
    if mid == "pos":
        if pre == "br":
            return "Bundesrat"
        if pre == "bv":
            return "Bundesvers."
    if pre == "p":
        return mid.upper()
    return f"{pre.upper()}-{mid.upper()}"

parties = [party_short(i) for i in df_norm.index]  # fehlte: Liste für ylabels

heatmap(
    df_norm,
    xlabel="Kantone",
    ylabel="Parteien",
    xlabels=cantons,
    ylabels=parties,
    figsize=(16, 6),
)

## Heatmap timeslot

In [ ]:
df_phase = pd.read_csv("../data/processed/df_heatmap_by_phase.csv")
if "Unnamed: 0" in df_phase.columns:
    df_phase = df_phase.drop(columns=["Unnamed: 0"])
df_phase.head()

In [ ]:
PHASES = [
    ("phase1_fruehphase", "Frühphase (1848–1899)"),
    ("phase2_volatile", "Volatile Phase (1900–1949)"),
    ("phase3_konsens", "Konsensphase (1950–1980)"),
    ("phase4_aufspaltung", "Aufspaltung (1981–2009)"),
    ("phase5_2010_heute", "2010er–heute"),
]


def party_short(name):
    s = str(name)
    if "-" not in s or "_" not in s:
        return s
    pre, rest = s.split("-", 1)
    mid = rest.split("_", 1)[0]
    if mid == "pos":
        if pre == "br":
            return "Bundesrat"
        if pre == "bv":
            return "Bundesvers."
    if pre == "p":
        return mid.upper()
    return f"{pre.upper()}-{mid.upper()}"

In [ ]:
for slug, titel in PHASES:
    sub = df_phase.loc[df_phase["phase"] == slug]
    if sub.empty:
        print(f"Übersprungen (keine Daten): {titel}")
        continue

    M = sub.set_index("partei").drop(columns=["phase"], errors="ignore").astype(float)
    span = M.max().max() - M.min().min()
    df_norm_phase = (M - M.min().min()) / span if span else M * 0

    cantons = [str(c)[:2].upper() for c in df_norm_phase.columns]
    parties = [party_short(i) for i in df_norm_phase.index]

    print(titel)
    heatmap(
        df_norm_phase,
        xlabel="Kantone",
        ylabel="Parteien",
        xlabels=cantons,
        ylabels=parties,
        figsize=(16, 6),
    )